In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTE, ADASYN
from xgboost import XGBClassifier


# LOADING THE DATA

df = pd.read_csv('/content/bank-additional-full.csv', sep=';', engine='python')
df = df.sample(n=10000, random_state=42)


# CLEANING

df.drop('duration', axis=1, inplace=True)

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

df['y'] = df['y'].map({'no': 0, 'yes': 1})
df = pd.get_dummies(df, drop_first=True)


# SPLITING

X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# SCALING

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)   # 🔥 FIXED


# SAMPLING

smote = SMOTE(random_state=42)
adasyn = ADASYN(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train, y_train)

# MODELS

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=3,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=16,
        min_samples_split=4,
        min_samples_leaf=2,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(120, 60),
        activation='relu',
        solver='adam',
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=220,
        max_depth=6,
        learning_rate=0.07,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1,
        reg_alpha=0.1,
        eval_metric='logloss',
        random_state=42
    )
}

# CROSS VALIDATION

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# FUNCTION

def evaluate_models(X, y, dataset_name):
    print(f"\n===== {dataset_name} =====\n")

    results = []

    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=skf, scoring=scoring)

        acc = round(np.mean(scores['test_accuracy']), 4)
        prec = round(np.mean(scores['test_precision']), 4)
        rec = round(np.mean(scores['test_recall']), 4)
        f1 = round(np.mean(scores['test_f1']), 4)
        roc = round(np.mean(scores['test_roc_auc']), 4)


        print(f"Model: {name}")
        print(f"Accuracy: {acc}")
        print(f"Precision: {prec}")
        print(f"Recall: {rec}")
        print(f"F1 Score: {f1}")
        print(f"ROC-AUC: {roc}")
        print("-" * 35)

        results.append({
            "Dataset": dataset_name,
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "ROC-AUC": roc
        })

    return pd.DataFrame(results)


df1 = evaluate_models(X_train, y_train, "Baseline")
df2 = evaluate_models(X_train_sm, y_train_sm, "SMOTE")
df3 = evaluate_models(X_train_ad, y_train_ad, "ADASYN")


# COMBINING OUTOUTS

final_df = pd.concat([df1, df2, df3], ignore_index=True)

dataset_order = ["Baseline", "SMOTE", "ADASYN"]
model_order = ["Decision Tree", "Random Forest", "MLP", "XGBoost"]

final_df["Dataset"] = pd.Categorical(final_df["Dataset"], categories=dataset_order, ordered=True)
final_df["Model"] = pd.Categorical(final_df["Model"], categories=model_order, ordered=True)

final_df = final_df.sort_values(by=["Dataset", "Model"])


top_models = final_df.sort_values(by="F1 Score", ascending=False).head(5)


final_df.to_excel("final_output.xlsx", index=False)


# FINAL Result

print("\n Saved: final_output.xlsx and best_models.xlsx\n")




===== Baseline =====

Model: Decision Tree
Accuracy: 0.8794
Precision: 0.441
Recall: 0.2362
F1 Score: 0.3061
ROC-AUC: 0.6201
-----------------------------------
Model: Random Forest
Accuracy: 0.8954
Precision: 0.6018
Recall: 0.2152
F1 Score: 0.3158
ROC-AUC: 0.7608
-----------------------------------
Model: MLP
Accuracy: 0.8924
Precision: 0.5838
Recall: 0.1918
F1 Score: 0.2857
ROC-AUC: 0.7243
-----------------------------------
Model: XGBoost
Accuracy: 0.8922
Precision: 0.5556
Recall: 0.2462
F1 Score: 0.3393
ROC-AUC: 0.7551
-----------------------------------

===== SMOTE =====

Model: Decision Tree
Accuracy: 0.8898
Precision: 0.9254
Recall: 0.848
F1 Score: 0.8849
ROC-AUC: 0.93
-----------------------------------
Model: Random Forest
Accuracy: 0.9303
Precision: 0.942
Recall: 0.917
F1 Score: 0.9293
ROC-AUC: 0.9782
-----------------------------------
Model: MLP
Accuracy: 0.9254
Precision: 0.9082
Recall: 0.9466
F1 Score: 0.9269
ROC-AUC: 0.9705
-----------------------------------
Model: XG